In [1]:
! pip install pandas matplotlib numpy tensorflow scikit-learn
! pip install nvidia-cudnn-cu12==9.3.0.75

In [2]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Input,
    Add,
    LayerNormalization
)

from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from boxe_utils import *
from stance_utils import *

I0000 00:00:1781664590.076313 1921338 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781664590.146628 1921338 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781664591.897723 1921338 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
def collapse_to_3_classes(label_6):
    mapping = {
        'Jab': 'Straight',
        'Cross': 'Straight',
        'Lead Hook': 'Hook',
        'Rear Hook': 'Hook',
        'Lead Uppercut': 'Uppercut',
        'Rear Uppercut': 'Uppercut',
    }
    return mapping[label_6]

In [4]:
TRAIN_VIDEOS = ["V1", "V2", "V7", "V8"]
CV_VIDEOS    = ["V3", "V4"]
TEST_VIDEOS  = ["V5", "V9"]

def load_concat(vids):
    Xs = []
    ys = []

    for v in vids:
        sk, lb = load_video(v)
        Xs.append(sk.astype(np.float32))
        ys.append(lb)

    return np.concatenate(Xs), np.concatenate(ys)

X_tr_all, y_tr_all = load_concat(TRAIN_VIDEOS)
X_cv, y_cv_raw = load_concat(CV_VIDEOS)
X_te, y_te_raw = load_concat(TEST_VIDEOS)

In [5]:
y_tr_all = np.array([collapse_to_3_classes(y) for y in y_tr_all])
y_cv_raw = np.array([collapse_to_3_classes(y) for y in y_cv_raw])
y_te_raw = np.array([collapse_to_3_classes(y) for y in y_te_raw])

classes, label_to_id, id_to_label = build_label_mapping(y_tr_all)

y_tr_all_id = np.array([label_to_id[x] for x in y_tr_all])
y_cv = np.array([label_to_id[x] for x in y_cv_raw])
y_te = np.array([label_to_id[x] for x in y_te_raw])

num_classes = len(classes)

In [6]:
tr_idx, val_idx = train_test_split(
    np.arange(len(y_tr_all_id)),
    test_size=0.15,
    stratify=y_tr_all_id,
    random_state=42
)

X_tr = X_tr_all[tr_idx]
y_tr = y_tr_all_id[tr_idx]

X_val = X_tr_all[val_idx]
y_val = y_tr_all_id[val_idx]

In [7]:
def axis_stats(X):

    B = body_frame_windows(X)

    x = B[...,0]
    y = B[...,1]

    mean = np.array([
        x[x != 0].mean(),
        y[y != 0].mean()
    ])

    std = np.array([
        x[x != 0].std() + 1e-6,
        y[y != 0].std() + 1e-6
    ])

    return mean.astype(np.float32), std.astype(np.float32)

mean, std = axis_stats(X_tr)

np.savez(
    "norm_stats.npz",
    mean=mean,
    std=std
)

In [8]:
def std_va(B):

    X = B.reshape(len(B), WINDOW_LEN, 34).copy()

    X[:,:,0::2] = (X[:,:,0::2] - mean[0]) / std[0]
    X[:,:,1::2] = (X[:,:,1::2] - mean[1]) / std[1]

    return add_velocity_and_acceleration(X)

def feats(X):
    return std_va(body_frame_windows(X))

In [9]:
SWAP_J = [
    (1,2),
    (3,4),
    (5,6),
    (7,8),
    (9,10),
    (11,12),
    (13,14),
    (15,16)
]

def flip_body(B):

    F = B.copy()

    F[...,0] *= -1

    for a,b in SWAP_J:
        tmp = F[:,:,a,:].copy()
        F[:,:,a,:] = F[:,:,b,:]
        F[:,:,b,:] = tmp

    return F

B_tr = body_frame_windows(X_tr)

X_tr_feat = np.concatenate([
    std_va(B_tr),
    std_va(flip_body(B_tr))
])

y_tr_final = np.concatenate([
    y_tr,
    y_tr
])

In [11]:
def build_model(input_shape, nc):

    inp = Input(shape=input_shape)

    x = Bidirectional(
        LSTM(
            64,
            return_sequences=True,
            dropout=0.3,
            recurrent_dropout=0.2
        )
    )(inp)

    x = Dropout(0.3)(x)

    att = MultiHeadAttention(
        num_heads=2,
        key_dim=32
    )(x,x)

    x = Add()([x,att])
    x = LayerNormalization()(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(5e-4)
    )(x)

    x = Dropout(0.3)(x)

    out = Dense(
        nc,
        activation="softmax"
    )(x)

    return Model(inp,out)

model = build_model(
    input_shape=X_tr_feat.shape[1:],
    nc=num_classes
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.1
    ),
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1781664734.018324 1921338 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 70285 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3, pci bus id: 0000:1b:00.0, compute capability: 9.0a
I0000 00:00:1781664734.553692 1921338 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 35414 MB memory:  -> device: 1, name: NVIDIA H100 80GB HBM3, pci bus id: 0000:43:00.0, compute capability: 9.0a
I0000 00:00:1781664734.877218 1921338 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 64987 MB memory:  -> device: 2, name: NVIDIA H100 80GB HBM3, pci bus id: 0000:52:00.0, compute capability: 9.0a
I0000 00:00:1781664735.763104 1921338 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 41257 MB memory:  -> device: 3, name: NVIDIA H100 80GB HBM3, pci bus id: 0000:61:00.0, compute capability: 9.0a
I0000 00:00:1781664736.339825 1921338 gpu_device.cc:2043] Cr

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 25, 102)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 25, 128)   │     85,504 │ input_layer[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 25, 128)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 25, 128)   │     33,088 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 25, 128)   │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 25, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3)         │        195 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 127,299 (497.26 KB)

 Trainable params: 127,299 (497.26 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
weights = compute_class_weight(
    "balanced",
    classes=np.unique(y_tr_final),
    y=y_tr_final
)

class_weights = dict(enumerate(weights))

In [13]:
callbacks = [

    EarlyStopping(
        monitor="val_accuracy",
        patience=15,
        restore_best_weights=True,
        mode="max"
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        mode="min"
    ),

    ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    )
]

In [14]:
y_tr_oh = tf.keras.utils.to_categorical(
    y_tr_final,
    num_classes
)

y_val_oh = tf.keras.utils.to_categorical(
    y_val,
    num_classes
)

model.fit(
    X_tr_feat,
    y_tr_oh,
    validation_data=(feats(X_val), y_val_oh),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 52s 246ms/step - accuracy: 0.4157 - loss: 1.1286 - val_accuracy: 0.5134 - val_loss: 1.0030 - learning_rate: 5.0000e-04
Epoch 2/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 39s 247ms/step - accuracy: 0.5392 - loss: 1.0286 - val_accuracy: 0.5535 - val_loss: 0.9989 - learning_rate: 5.0000e-04
Epoch 3/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 36s 236ms/step - accuracy: 0.6043 - loss: 0.9657 - val_accuracy: 0.6872 - val_loss: 0.8396 - learning_rate: 5.0000e-04
Epoch 4/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 37s 240ms/step - accuracy: 0.6280 - loss: 0.9209 - val_accuracy: 0.7299 - val_loss: 0.7735 - learning_rate: 5.0000e-04
Epoch 5/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 35s 226ms/step - accuracy: 0.6789 - loss: 0.8744 - val_accuracy: 0.7647 - val_loss: 0.7471 - learning_rate: 5.0000e-04
Epoch 6/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 37s 233ms/step - accuracy: 0.7125 - loss: 0.8236 - val_accuracy: 0.8209 - val_loss: 0.6887 - learning_rate: 5.0000e-04
Epoch 7/100
133/133 ━━━━━━━━━━━━━━━━━━━━

In [15]:
from tensorflow.keras.models import load_model

best_model = load_model("best_model.keras")
y_pred = np.argmax(best_model.predict(X_te_feat), axis=1)

print(classification_report(y_te, y_pred, target_names=classes_3))

cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=classes,
            yticklabels=classes,
            cmap='Blues')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.title(f'Matriz de Confusão {TEST_VIDEOS}')
plt.tight_layout()
plt.savefig("matrix.png")
plt.show()


NameError: name 'X_te_feat' is not defined